# Comparison — 4 Models (LightGBM, CatBoost, XGBoost, Random Forest)
Obesity Risk Estimation | Coding Week 2026

In [ ]:
# Install required packages
!pip install catboost lightgbm xgboost scikit-learn pandas numpy matplotlib seaborn joblib

In [ ]:
import os
import warnings
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score)

warnings.filterwarnings('ignore')

CLASSES = ['Insufficient_Weight', 'Normal_Weight', 'Obesity_Type_I', 'Obesity_Type_II', 'Obesity_Type_III', 'Overweight_Level_I', 'Overweight_Level_II']
CLASSES_SHORT = ['Ins', 'Nor', 'Ob1', 'Ob2', 'Ob3', 'Ow1', 'Ow2']

# Create outputs directory
os.makedirs('../outputs', exist_ok=True)
print("Outputs directory ready")

In [ ]:
# Load data
print("LOADING DATA")
X_train = pd.read_csv('../data/X_train.csv')
X_test  = pd.read_csv('../data/X_test.csv')
y_train = pd.read_csv('../data/y_train.csv').squeeze()
y_test  = pd.read_csv('../data/y_test.csv').squeeze()
X_test = X_test[X_train.columns]
print(f"Train: {X_train.shape[0]} samples, Test: {X_test.shape[0]} samples, Features: {X_train.shape[1]}")

In [ ]:
# Train all 4 models
results = {}
models = {}

# 1. LightGBM
print("[1/4] Training LightGBM...")
lgbm = LGBMClassifier(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42, verbose=-1)
lgbm.fit(X_train, y_train)
y_pred = lgbm.predict(X_test)
y_prob = lgbm.predict_proba(X_test)
results["LightGBM"] = {"accuracy": accuracy_score(y_test, y_pred), "precision": precision_score(y_test, y_pred, average="weighted"), "recall": recall_score(y_test, y_pred, average="weighted"), "f1": f1_score(y_test, y_pred, average="weighted"), "roc_auc": roc_auc_score(y_test, y_prob, multi_class="ovr"), "y_pred": y_pred}
models["LightGBM"] = lgbm
joblib.dump(lgbm, '../outputs/lightgbm.pkl')
print("Done")

In [ ]:
# 2. CatBoost
print("[2/4] Training CatBoost...")
cat = CatBoostClassifier(iterations=500, learning_rate=0.1, depth=6, random_seed=42, verbose=0)
cat.fit(X_train, y_train)
y_pred = cat.predict(X_test)
y_prob = cat.predict_proba(X_test)
results["CatBoost"] = {"accuracy": accuracy_score(y_test, y_pred), "precision": precision_score(y_test, y_pred, average="weighted"), "recall": recall_score(y_test, y_pred, average="weighted"), "f1": f1_score(y_test, y_pred, average="weighted"), "roc_auc": roc_auc_score(y_test, y_prob, multi_class="ovr", average="weighted"), "y_pred": y_pred}
models["CatBoost"] = cat
joblib.dump(cat, '../outputs/catboost.pkl')
print("Done")

In [ ]:
# 3. XGBoost
print("[3/4] Training XGBoost...")
xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, eval_metric='mlogloss', random_state=42, use_label_encoder=False)
xgb.fit(X_train, y_train)
y_pred = xgb.predict(X_test)
y_prob = xgb.predict_proba(X_test)
results["XGBoost"] = {"accuracy": accuracy_score(y_test, y_pred), "precision": precision_score(y_test, y_pred, average="weighted"), "recall": recall_score(y_test, y_pred, average="weighted"), "f1": f1_score(y_test, y_pred, average="weighted"), "roc_auc": roc_auc_score(y_test, y_prob, multi_class="ovr"), "y_pred": y_pred}
models["XGBoost"] = xgb
joblib.dump(xgb, '../outputs/xgboost.pkl')
print("Done")

In [ ]:
# 4. Random Forest
print("[4/4] Training Random Forest...")
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)
results["Random Forest"] = {"accuracy": accuracy_score(y_test, y_pred), "precision": precision_score(y_test, y_pred, average="weighted"), "recall": recall_score(y_test, y_pred, average="weighted"), "f1": f1_score(y_test, y_pred, average="weighted"), "roc_auc": roc_auc_score(y_test, y_prob, multi_class="ovr"), "y_pred": y_pred}
models["Random Forest"] = rf
joblib.dump(rf, '../outputs/random_forest.pkl')
print("Done")

In [ ]:
# Display metrics comparison
print("\n" + "="*70)
print("MODEL COMPARISON - METRICS")
print("="*70)
print(f"\n{'Model':<15} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1-Score':>10} {'ROC-AUC':>10}")
print("-"*70)
for name, m in results.items():
    print(f"{name:<15} {m['accuracy']*100:>9.2f}% {m['precision']*100:>9.2f}% {m['recall']*100:>9.2f}% {m['f1']*100:>9.2f}% {m['roc_auc']:>9.4f}")
best = max(results, key=lambda n: results[n]["accuracy"])
print(f"\nBest (Accuracy): {best} ({results[best]['accuracy']*100:.2f}%)")
best_roc = max(results, key=lambda n: results[n]["roc_auc"])
print(f"Best (ROC-AUC): {best_roc} ({results[best_roc]['roc_auc']:.4f})")

In [ ]:
# Classification reports
for name, m in results.items():
    print(f"\n{name}")
    print(classification_report(y_test, m["y_pred"], target_names=CLASSES))

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle("Confusion Matrices", fontsize=16, fontweight="bold")
colors = ["#3498DB", "#E74C3C", "#2ECC71", "#9B59B6"]
for idx, (name, m) in enumerate(results.items()):
    ax = axes[idx//2, idx%2]
    sns.heatmap(confusion_matrix(y_test, m["y_pred"]), annot=True, fmt="d", cmap="Blues", ax=ax, xticklabels=CLASSES_SHORT, yticklabels=CLASSES_SHORT, cbar=False)
    ax.set_title(f"{name}\nAcc: {m['accuracy']*100:.1f}%", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig('../outputs/confusion_matrices.png', dpi=150)
plt.show()
print("Saved: confusion_matrices.png")

In [ ]:
# Metrics comparison bars
fig, ax = plt.subplots(figsize=(12, 6))
model_names = list(results.keys())
x = np.arange(len(model_names))
width = 0.15
metrics_list = [("Accuracy", "accuracy"), ("Precision", "precision"), ("Recall", "recall"), ("F1", "f1"), ("ROC-AUC", "roc_auc")]
bar_colors = ["#3498DB", "#E74C3C", "#2ECC71", "#9B59B6", "#F39C12"]
for i, (label, metric) in enumerate(metrics_list):
    values = [results[n][metric]*100 for n in model_names]
    ax.bar(x + i*width - 2*width, values, width, label=label, color=bar_colors[i])
ax.set_ylabel("Score (%)")
ax.set_title("All Metrics Comparison")
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.legend()
ax.set_ylim(0, 110)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/all_metrics_comparison.png', dpi=150)
plt.show()
print("Saved: all_metrics_comparison.png")

In [ ]:
# Feature importance
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
fig.suptitle("Feature Importance", fontsize=16, fontweight="bold")
for idx, (name, model) in enumerate(models.items()):
    ax = axes[idx//2, idx%2]
    if hasattr(model, 'feature_importances_'): imp = model.feature_importances_
    elif hasattr(model, 'get_feature_importance'): imp = model.get_feature_importance()
    else: imp = np.zeros(len(X_train.columns))
    imp_df = pd.DataFrame({'Feature': X_train.columns, 'Importance': imp}).sort_values('Importance', ascending=True)
    ax.barh(imp_df['Feature'], imp_df['Importance'], color=colors[idx])
    ax.set_title(f"{name}", fontweight="bold")
plt.tight_layout()
plt.savefig('../outputs/feature_importance_comparison.png', dpi=150)
plt.show()
print("Saved: feature_importance_comparison.png")

In [ ]:
# List saved files
import os
print("\nFiles saved to outputs/:")
for f in os.listdir('../outputs'):
    print(f"  - {f}")